# 00 — Preparação da base

## Objetivo

Como transformar a base original em uma versão curta e legível para o curso?

Este notebook valida o contrato mínimo, traduz as colunas e salva uma cópia
preparada. O CSV bruto é somente lido.

In [1]:
from pathlib import Path
import sys

ponto_atual = Path.cwd().resolve()
RAIZ = next(
    caminho for caminho in (ponto_atual, *ponto_atual.parents)
    if (caminho / "data" / "raw" / "UCI_Credit_Card.csv").exists()
)
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))


import hashlib
import pandas as pd

from src.auxiliares import ALVO, COLUNA_ID, MAPEAMENTO_COLUNAS
from src.visual_utils import grafico_distribuicao_alvo

CAMINHO_BRUTO = RAIZ / "data" / "raw" / "UCI_Credit_Card.csv"
CAMINHO_PREPARADO = RAIZ / "data" / "processed" / "cartao_credito_portugues.csv"

## Como é a base?

A unidade observada é um registro por ID de cliente/conta. Não há datas por
linha: os meses aparecem apenas no significado das colunas.

In [2]:
hash_bruto = hashlib.sha256(CAMINHO_BRUTO.read_bytes()).hexdigest()
dados_brutos = pd.read_csv(CAMINHO_BRUTO)

print(f"Dimensões: {dados_brutos.shape[0]:,} linhas x {dados_brutos.shape[1]} colunas")
print(f"SHA-256: {hash_bruto}")
dados_brutos.head()

Dimensões: 30,000 linhas x 25 colunas
SHA-256: a0f0ab49d6326671d6cd83be5c88dcf18007025fe9a53ecd699119c871176ca1


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,1,20000.0,2,2,1,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,2,2,2,34,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,2,2,1,37,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,1,2,1,57,-1,0,-1,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0


## O contrato permite modelagem?

Antes de qualquer modelo, verificamos chave, granularidade, missing, duplicados
e outcome. A base não contém tratamento nem controle; este projeto é preditivo,
não causal.

In [3]:
alvo_original = "default.payment.next.month"
resumo_qualidade = pd.Series({
    "linhas": len(dados_brutos),
    "colunas": dados_brutos.shape[1],
    "missing": int(dados_brutos.isna().sum().sum()),
    "duplicatas_exatas": int(dados_brutos.duplicated().sum()),
    "ids_duplicados": int(dados_brutos["ID"].duplicated().sum()),
    "taxa_inadimplencia": dados_brutos[alvo_original].mean(),
})
assert dados_brutos["ID"].is_unique
assert set(dados_brutos[alvo_original].unique()) == {0, 1}
resumo_qualidade.to_frame("resultado")

,resultado
linhas,30000.0000
colunas,25.0000
missing,0.0000
duplicatas_exatas,0.0000
ids_duplicados,0.0000
taxa_inadimplencia,0.2212


Os 30 mil IDs são únicos, não há valores ausentes e o target é binário.
Perfis coincidentes sem o ID são mantidos: eles não comprovam duplicação do
mesmo cliente.

## Como ficam os nomes em português?

Os valores e códigos permanecem iguais aos da fonte. Somente os nomes mudam.

In [4]:
dicionario_colunas = MAPEAMENTO_COLUNAS.copy()
pd.DataFrame(
    dicionario_colunas.items(),
    columns=["nome_original", "nome_no_curso"],
)

,nome_original,nome_no_curso
0,ID,id_cliente
1,LIMIT_BAL,limite_credito
2,SEX,sexo
3,EDUCATION,escolaridade
4,MARRIAGE,estado_civil
5,AGE,idade
6,PAY_0,status_pagamento_set
7,PAY_2,status_pagamento_ago
8,PAY_3,status_pagamento_jul
9,PAY_4,status_pagamento_jun


In [5]:
dados = dados_brutos.rename(columns=dicionario_colunas)
CAMINHO_PREPARADO.parent.mkdir(parents=True, exist_ok=True)
dados.to_csv(CAMINHO_PREPARADO, index=False)

assert dados.shape == dados_brutos.shape
assert COLUNA_ID in dados and ALVO in dados
print(f"Base preparada salva em: {CAMINHO_PREPARADO}")

Base preparada salva em: C:\GitHub\ALURA-TESTE\classificacao-avancando-classificadores\data\processed\cartao_credito_portugues.csv


## Qual é o desbalanceamento do problema?

A classe positiva é minoritária, mas ainda possui milhares de exemplos.

In [6]:
distribuicao_alvo = (
    dados[ALVO].value_counts().sort_index()
    .rename_axis("inadimplente").to_frame("clientes")
)
distribuicao_alvo["proporcao"] = distribuicao_alvo["clientes"] / len(dados)

display(distribuicao_alvo)
fig = grafico_distribuicao_alvo(distribuicao_alvo)
fig.show()

,clientes,proporcao
inadimplente,,
0,23364,0.7788
1,6636,0.2212


## Existem códigos ou valores que exigem cuidado?

Os códigos não documentados são registrados, não corrigidos. Valores
monetários negativos podem representar saldo credor e também são preservados.

In [7]:
colunas_categoricas = ["sexo", "escolaridade", "estado_civil"]
codigos = pd.Series({
    coluna: sorted(dados[coluna].unique().tolist())
    for coluna in colunas_categoricas
}, name="valores_observados")

faixas = dados[
    ["limite_credito", "idade", "valor_fatura_set", "valor_pago_set"]
].agg(["min", "median", "max"]).T
display(codigos.to_frame(), faixas)

,valores_observados
sexo,"[1, 2]"
escolaridade,"[0, 1, 2, 3, 4, 5, 6]"
estado_civil,"[0, 1, 2, 3]"


,min,median,max
limite_credito,10000.0,140000.0,1000000.0
idade,21.0,34.0,79.0
valor_fatura_set,-165580.0,22381.5,964511.0
valor_pago_set,0.0,2100.0,873552.0


## Resultado

A cópia em português preserva 30.000 linhas e 25 colunas. O ID será excluído
apenas das features; inadimplente será o outcome. A taxa positiva é 22,12%.